# Exploratory data analysis and possible feature engineering

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Import the training dataset
TRAIN_DATA_PATH = Path("data/train_data.parquet")

train_df = pd.read_parquet(TRAIN_DATA_PATH)

train_df.index = pd.to_numeric(train_df.index)

In [ ]:
# Convert all "Int64" type columns to "float64" for EDA
train_eda = train_df.copy()
int64_cols = train_eda.select_dtypes(include=["Int64"]).columns
train_eda[int64_cols] = train_eda[int64_cols].astype(float)

## Exploratory data analysis with packages ydata_profiling and sweetviz

ydata_profiling created too buig unloadable html, so making twi different

In [3]:
from data_profiling import ProfileReport

# Select the 50 features with the most filled values (non-missing)
# and include the target columns
targets = ["target", "target_annual_roi"]

top_50_features = (
    train_eda.drop(columns=targets, errors="ignore")
    .notna()
    .sum()
    .nlargest(50)
    .index.tolist()
)

final_cols = top_50_features + targets

train_eda_50 = train_eda[final_cols].copy()

# Generate the data profiling report for the selected features and targets
profile = ProfileReport(
    train_eda_50,
    title="EDA: LendingClub (50 most filled features + targets)",
    explorative=True,
    correlations={
        "auto": {"calculate": True},
        "phi_k": {"calculate": True}
    },
)

# Save the data profiling report as an HTML file
DATA_PROFILING_TOP50_PATH = Path("eda/data_profiling_top50.html")
profile.to_file(DATA_PROFILING_TOP50_PATH)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 52/52 [00:03<00:00, 15.46it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Generate the data profiling report for all features and targets
# with no interactions
profile = ProfileReport(
    train_eda,
    title="EDA: LendingClub (all features)",
    explorative=True,
    interactions=None,
    correlations={
        "auto": {"calculate": True},
        "phi_k": {"calculate": True},
    },
)

DATA_PROFILING_ALL_PATH = Path("eda/data_profiling_all.html")
profile.to_file(DATA_PROFILING_ALL_PATH)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 95/95 [00:12<00:00,  7.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import sweetviz as sv

# Generate the Sweetviz report for all features and targets
report_sv = sv.analyze(train_eda, target_feat="target")

SWEETVIZ_ALL_PATH = Path("eda/sweetviz.html")
report_sv.show_html(SWEETVIZ_ALL_PATH, open_browser=False)

                                             |          | [  0%]   00:00 -> (? left)

Report eda/sweetviz.html was generated.


We will handle the high cardinality of the addr_state column by applying a transformation for models that cannot process categorical features. In cases of high correlation, we will evaluate dropping the respective columns. Zero and skewed values do not negatively impact GBM based models, and zero and missing values reflect real-world conditions.

## Find possible new features to add with OpenFE

In [ ]:
import os
from openfe import OpenFE, get_candidate_features

# Split the training data into features and target
target_cols = ["target", "target_annual_roi"]
X_train = train_df.drop(columns=target_cols)
y_train = train_df["target"]

# Drop datetime columns from the features
datetime_cols = X_train.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()
X_train = X_train.drop(columns=datetime_cols)

# Identify categorical and numerical columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

all_candidates = get_candidate_features(
    numerical_features=num_cols, categorical_features=cat_cols
)

# Filter candidate features to include only those with specific operations
allowed_operations = ["/", "*", "groupby"]
filtered_candidates = [
    f for f in all_candidates if any(op in f.name for op in allowed_operations)
]

# Initialize OpenFE and fit it to the training data with the filtered candidate
# features and use all available CPU cores minus one for parallel processing
ofe = OpenFE()
n_cores = max(1, os.cpu_count() - 1)

features_cat = ofe.fit(
    data=X_train.tail(300000),
    label=y_train.tail(300000),
    n_jobs=n_cores,
    candidate_features_list=filtered_candidates,
    verbose=False,
    seed=42,
)

In [4]:
# Function to recursively get the formula of a feature from its tree structure
def get_formula(node):
    if not hasattr(node, "children") or not node.children:
        return getattr(node, "name", str(node))

    if len(node.children) == 2:
        left = get_formula(node.children[0])
        right = get_formula(node.children[1])
        if node.name.startswith("GroupByThen"):
            stat_type = node.name.split("Then")[1]
            return f"{stat_type}({left} grouped by {right})"
        return f"({left} {node.name} {right})"

    if len(node.children) == 1:
        return f"{node.name}({get_formula(node.children[0])})"

    return node.name


# Display the top 100 generated features and their formulas
# and save the candidate features for later evaluation
top_n = 100
top_features_cat = features_cat[:top_n]

print(f"Totally generated features: {len(features_cat)}")
print(f"Best {top_n} features:\n")

candidate_features = {}

for i, f in enumerate(top_features_cat):
    formula = get_formula(f)
    print(f"Order #{i+1} | Formula: {formula}")

    if hasattr(f, "children") and len(f.children) == 2:
        op = f.name
        if op in ["/", "*"]:
            col1 = f.children[0].name
            col2 = f.children[1].name

            op_str = "div" if op == "/" else "x"
            feat_name = f"fe_{col1}_{op_str}_{col2}"

            candidate_features[feat_name] = (col1, op, col2)

Totally generated features: 2000
Best 100 features:

Order #1 | Formula: (sub_grade * term_months)
Order #2 | Formula: (int_rate / fico_avg)
Order #3 | Formula: (int_rate * term_months)
Order #4 | Formula: (fico_avg / term_months)
Order #5 | Formula: (sub_grade * num_actv_rev_tl)
Order #6 | Formula: (sub_grade * dti)
Order #7 | Formula: (sub_grade / fico_avg)
Order #8 | Formula: (sub_grade / avg_cur_bal)
Order #9 | Formula: (loan_amnt / annual_inc)
Order #10 | Formula: (dti / avg_cur_bal)
Order #11 | Formula: (dti / mort_acc)
Order #12 | Formula: (dti * term_months)
Order #13 | Formula: (sub_grade * acc_open_past_24mths)
Order #14 | Formula: (acc_open_past_24mths / tot_hi_cred_lim)
Order #15 | Formula: (loan_amnt / total_rev_hi_lim)
Order #16 | Formula: (total_rev_hi_lim / num_actv_rev_tl)
Order #17 | Formula: (total_rev_hi_lim / num_rev_tl_bal_gt_0)
Order #18 | Formula: (acc_open_past_24mths / mort_acc)
Order #19 | Formula: (loan_amnt * sub_grade)
Order #20 | Formula: (acc_open_past_2

In [5]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit, cross_validate


# Function to safely perform division and handle division by zero by replacing it with NaN
def safe_divide(df, col_num, col_den):
    return df[col_num] / df[col_den].replace(0, np.nan)


# Function to evaluate the model using time series cross-validation and return the mean AUC score
def evaluate_cv(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    clf = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
    results = cross_validate(clf, X, y, cv=tscv, scoring="roc_auc")
    return np.mean(results["test_score"])


# Evaluate the baseline model with the original features to get the initial AUC score
current_best_auc = evaluate_cv(X_train, y_train)
print(f"Baseline Val AUC: {current_best_auc:.5f}\n")

# Iterate through the candidate features, create them in the training data, evaluate the model,
# and keep track of accepted features based on AUC improvement
X_best = X_train.copy()
accepted_features = []

for feat_name, (col1, op, col2) in candidate_features.items():

    X_temp = X_best.copy()
    if op == "/":
        X_temp[feat_name] = safe_divide(X_temp, col1, col2)
    elif op == "*":
        X_temp[feat_name] = X_temp[col1] * X_temp[col2]

    new_auc = evaluate_cv(X_temp, y_train)
    improvement = new_auc - current_best_auc
    if improvement > 0.0001:
        print(
            f"ACCEPTED: {feat_name} | AUC raised by: {improvement:.5f} (New: {new_auc:.5f})"
        )
        X_best = X_temp.copy()
        current_best_auc = new_auc
        accepted_features.append(feat_name)
    else:
        print(f"REJECTED: {feat_name} | AUC changed by: {improvement:.5f}")


print(f"Used {len(accepted_features)} Features:")
for f in accepted_features:
    print(f" - {f}")

Baseline Val AUC: 0.73040

ACCEPTED: fe_sub_grade_x_term_months | AUC raised by: 0.00019 (New: 0.73060)
ACCEPTED: fe_int_rate_div_fico_avg | AUC raised by: 0.00011 (New: 0.73070)
REJECTED: fe_int_rate_x_term_months | AUC changed by: -0.00050
REJECTED: fe_fico_avg_div_term_months | AUC changed by: -0.00004
REJECTED: fe_sub_grade_x_num_actv_rev_tl | AUC changed by: -0.00017
REJECTED: fe_sub_grade_x_dti | AUC changed by: -0.00021
REJECTED: fe_sub_grade_div_fico_avg | AUC changed by: -0.00041
REJECTED: fe_sub_grade_div_avg_cur_bal | AUC changed by: -0.00051
REJECTED: fe_loan_amnt_div_annual_inc | AUC changed by: -0.00066
REJECTED: fe_dti_div_avg_cur_bal | AUC changed by: -0.00033
REJECTED: fe_dti_div_mort_acc | AUC changed by: -0.00007
REJECTED: fe_dti_x_term_months | AUC changed by: -0.00022
REJECTED: fe_sub_grade_x_acc_open_past_24mths | AUC changed by: -0.00033
REJECTED: fe_acc_open_past_24mths_div_tot_hi_cred_lim | AUC changed by: -0.00011
REJECTED: fe_loan_amnt_div_total_rev_hi_lim | 

Features are not useful and if yes by very small margin so we will use only existing features which are already 92 + 2 target columns.

In [ ]:
# Possible code for adding of accepted features to the training and test sets
"""
TEST_DATA_PATH = Path("data/test_data.parquet")

test_df = pd.read_parquet(TEST_DATA_PATH)

test_df.index = pd.to_numeric(test_df.index)

train_df_fe = train_df.copy()
test_df_fe = test_df.copy()

for feat_name in accepted_features:
    col1, op, col2 = candidate_features[feat_name]

    if op == "/":
        train_df_fe[feat_name] = safe_divide(train_df_fe, col1, col2)
        test_df_fe[feat_name] = safe_divide(test_df_fe, col1, col2)
    elif op == "*":
        train_df_fe[feat_name] = train_df_fe[col1] * train_df_fe[col2]
        test_df_fe[feat_name] = test_df_fe[col1] * test_df_fe[col2]
"""

## Look for possible improvement if we drop existing features

In [6]:
# Not adding features, so dropping them
X_best = X_best.drop(columns=accepted_features)

# Evaluate model baseline with all original features
current_best_auc = evaluate_cv(X_best, y_train)
print(f"Baseline Val AUC: {current_best_auc:.5f}\n")

# Sort features by importance using a LightGBM model
clf_imp = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
clf_imp.fit(X_best, y_train)

importances = pd.Series(
    clf_imp.feature_importances_, index=X_best.columns
).sort_values()
features_to_test = importances.index.tolist()

# Iteratively drop features and evaluate the model,
dropped_features = []

for feat_name in features_to_test:
    X_temp = X_best.drop(columns=[feat_name])
    new_auc = evaluate_cv(X_temp, y_train)
    improvement = new_auc - current_best_auc

    if improvement > 0.00005:
        print(
            f"DELETED: {feat_name} | AUC improved by {improvement:.5f} (New: {new_auc:.5f})"
        )
        X_best = X_temp.copy()
        current_best_auc = new_auc
        dropped_features.append(feat_name)
    else:
        print(f"KEPT: {feat_name} | AUC changed by: {improvement:.5f}")

print(f"Final Val AUC: {current_best_auc:.5f}")
print(
    f"From the original {X_best.shape[1]} features, {len(dropped_features)} were deleted."
)

Baseline Val AUC: 0.73040

KEPT: issue_d_month_sin | AUC changed by: -0.00009
KEPT: earliest_cr_line_month_cos | AUC changed by: -0.00007
KEPT: chargeoff_within_12_mths | AUC changed by: -0.00005
KEPT: delinq_amnt | AUC changed by: -0.00003
KEPT: num_tl_90g_dpd_24m | AUC changed by: -0.00015
KEPT: application_type | AUC changed by: 0.00000
KEPT: tax_liens | AUC changed by: -0.00003
KEPT: acc_now_delinq | AUC changed by: -0.00000
KEPT: earliest_cr_line_month | AUC changed by: -0.00002
KEPT: num_op_rev_tl | AUC changed by: 0.00000
KEPT: disbursement_method | AUC changed by: 0.00000
KEPT: num_tl_30dpd | AUC changed by: 0.00000
KEPT: open_il_12m | AUC changed by: 0.00000
KEPT: num_accts_ever_120_pd | AUC changed by: -0.00012
KEPT: tot_coll_amt | AUC changed by: -0.00004
KEPT: open_il_24m | AUC changed by: 0.00000
DELETED: open_acc | AUC improved by 0.00013 (New: 0.73053)
KEPT: pub_rec | AUC changed by: 0.00000
KEPT: inq_last_12m | AUC changed by: 0.00000
KEPT: num_sats | AUC changed by: -0

No significant improvement when dropping so keeping all features and not changing the dataset

In [ ]:
# Possible code for saving feature engineered datasets after adding and dropping features
"""
train_df_fe = train_df_fe.drop(columns=dropped_features, errors="ignore")
test_df_fe = test_df_fe.drop(columns=dropped_features, errors="ignore")

train_df_fe.index = train_df_fe.index.astype(str)
test_df_fe.index = test_df_fe.index.astype(str)

TRAIN_DATA_FE_PATH = Path("data/train_data_fe.parquet")
TEST_DATA_FE_PATH = Path("data/test_data_fe.parquet")
train_df_fe.to_parquet(TRAIN_DATA_FE_PATH, index=False)
test_df_fe.to_parquet(TEST_DATA_FE_PATH, index=False)

train_df_fe.index = pd.to_numeric(train_df_fe.index)
test_df_fe.index = pd.to_numeric(test_df_fe.index)
"""